# Amparo -- M2: Evaluacion del modelo fine-tuneado

Este notebook corre el harness de evaluacion de M2 sobre el adaptador LoRA
que entreno M1 (`baseline_finetune.ipynb`, ya publicado en la wiki). No
modifica ni depende de que M1 haya dejado archivos guardados: vuelve a
generar las respuestas baseline y fine-tuned sobre el **mismo split de
validacion** (mismo seed) para que los numeros sean comparables con el
baseline ya publicado (3.4% / 17.7% de similitud lexica).

Fases: generar respuestas (baseline y fine-tuned) -> liberar el adaptador y
reusar el modelo base como juez -> LLM-as-judge + sondeo de position bias ->
metricas clasicas + metrica de dominio juridico (sin GPU) -> sesgos de
longitud y auto-preferencia -> scorecard, persistido en Drive.

Toda la logica pesada vive en `tools/evaluation/` (repo principal, no en
este notebook) -- este notebook solo clona el repo, instala dependencias, y
orquesta las llamadas.

Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de
ejecucion > GPU`.


In [ ]:
# Clona el repo (o actualiza si ya existe de una corrida anterior en esta VM)
import os

if not os.path.isdir("Amparo"):
    !git clone https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git pull

%cd Amparo


In [ ]:
# Dependencias puras de tools/evaluation (metricas clasicas, tests)
!pip install -q -r requirements.txt


In [ ]:
# Stack de ML pesado -- igual que en M1 (baseline_finetune.ipynb, celda 1):
# se instala aqui y NO en requirements.txt del repo, para no arriesgar
# reemplazar el build de PyTorch con CUDA que Colab ya trae preinstalado.
# bert-score va aqui tambien (no en requirements.txt) porque depende de
# torch de forma transitiva.
!pip install -q -U transformers peft bitsandbytes accelerate bert-score
!pip install -U "bitsandbytes>=0.46.1"


## Configuracion

Las constantes (seed, val_fraction, modelo base, rutas de Drive) viven en
`tools/evaluation/config.py` -- deben coincidir con `RANDOM_SEED=42` y
`VAL_FRACTION=0.15` de M1 para evaluar sobre el mismo split.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

from tools.evaluation import config, dataset

records = dataset.load_records()
train_records, val_records = dataset.stratified_split(records)
system_prompt = dataset.system_prompt(records)

print(f"Total: {len(records)} | train: {len(train_records)} | val: {len(val_records)}")
assert len(val_records) == 201, "El split no coincide con el de M1 -- revisar RANDOM_SEED/VAL_FRACTION"
print("Split verificado: coincide con el usado en M1.")


## Fase 1 -- Generacion: baseline (modelo sin fine-tuning)

Puede tardar varios minutos segun el tamano de la validacion (201 ejemplos).


In [ ]:
from tools.evaluation import generation

model, tokenizer = generation.load_base_model()

baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="baseline"
)
print(f"Generadas {len(baseline_results)} respuestas baseline.")


## Fase 2 -- Generacion: fine-tuned (adaptador LoRA desde Drive)


In [ ]:
model = generation.attach_adapter(model, config.DRIVE_ADAPTER_DIR)
model.eval()

finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="fine_tuned"
)
print(f"Generadas {len(finetuned_results)} respuestas fine-tuned.")


## Fase 3 -- Liberar el adaptador y reutilizar el modelo base como juez

`detach_adapter` usa `model.unload()` (quita las capas LoRA sin fusionarlas)
-- así el mismo modelo cargado en memoria sirve como juez independiente del
adaptador, sin una segunda carga completa de modelo.


In [ ]:
import torch

model = generation.detach_adapter(model)
torch.cuda.empty_cache()
print("Adaptador liberado. El modelo en memoria es ahora el juez (base, sin fine-tuning).")


## Fase 4 -- LLM-as-a-Judge

Puntuacion absoluta (1-5 por criterio) de cada respuesta contra la
referencia del dataset, mas un sub-experimento de *position bias* sobre una
muestra de 30 pares baseline/fine-tuned.


In [ ]:
from tools.evaluation import judge

judge_baseline = judge.score_batch(model, tokenizer, baseline_results)
judge_finetuned = judge.score_batch(model, tokenizer, finetuned_results)

n_fail_base = sum(1 for s in judge_baseline if not s.parse_ok)
n_fail_ft = sum(1 for s in judge_finetuned if not s.parse_ok)
print(f"Juez baseline: {len(judge_baseline)} filas, {n_fail_base} fallos de parseo.")
print(f"Juez fine-tuned: {len(judge_finetuned)} filas, {n_fail_ft} fallos de parseo.")


In [ ]:
from tools.evaluation import bias

pairs = [
    (b.id, b.query, b.generated, f.generated)
    for b, f in zip(baseline_results, finetuned_results)
]
position_bias_report = bias.run_position_bias_probe(model, tokenizer, pairs)
print(position_bias_report)


## Fase 5 -- Metricas clasicas y metrica de dominio juridico

No requieren GPU (BERTScore descarga un modelo aparte, chico, en CPU/GPU
segun disponibilidad).


In [ ]:
from tools.evaluation import pipeline

eval_rows_baseline = pipeline.build_eval_rows(baseline_results, judge_baseline)
eval_rows_finetuned = pipeline.build_eval_rows(finetuned_results, judge_finetuned)

pipeline.fill_bertscore(eval_rows_baseline)
pipeline.fill_bertscore(eval_rows_finetuned)

all_rows = eval_rows_baseline + eval_rows_finetuned
print(f"Total de filas evaluadas: {len(all_rows)}")


## Fase 6 -- Sesgos: length bias y self-preference bias


In [ ]:
length_bias_baseline = bias.length_bias_correlation(
    [r.judge_composite for r in eval_rows_baseline if r.judge_composite is not None],
    [len(r.generated) for r in eval_rows_baseline if r.judge_composite is not None],
)
length_bias_finetuned = bias.length_bias_correlation(
    [r.judge_composite for r in eval_rows_finetuned if r.judge_composite is not None],
    [len(r.generated) for r in eval_rows_finetuned if r.judge_composite is not None],
)

self_pref_report = bias.self_preference_gap(
    judge_baseline=[r.judge_composite for r in eval_rows_baseline if r.judge_composite is not None],
    judge_finetuned=[r.judge_composite for r in eval_rows_finetuned if r.judge_composite is not None],
    sim_baseline=[r.similarity_pct for r in eval_rows_baseline if r.similarity_pct is not None],
    sim_finetuned=[r.similarity_pct for r in eval_rows_finetuned if r.similarity_pct is not None],
)

bias_summary = {
    "length_bias_pearson_r_baseline": length_bias_baseline["pearson_r"],
    "length_bias_pearson_r_fine_tuned": length_bias_finetuned["pearson_r"],
    "self_preference_judge_gap": self_pref_report.judge_gap,
    "self_preference_similarity_gap": self_pref_report.similarity_gap,
    "self_preference_divergence": self_pref_report.divergence,
    "self_preference_flagged": self_pref_report.flagged,
    "position_bias_flip_rate_pct": position_bias_report.flip_rate_pct,
    "position_bias_n_pairs": position_bias_report.n_pairs,
}
bias_summary


## Fase 7 -- Scorecard y persistencia en Drive

Cada corrida se guarda en una carpeta con timestamp propio (no se sobreescribe
la corrida anterior mientras se itera).


In [ ]:
import json as _json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from tools.evaluation import scorecard

manifest = pipeline.build_manifest(n_val=len(val_records), hardware=subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip())

run_id = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")
drive_run_dir = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/{run_id}")
drive_run_dir.mkdir(parents=True, exist_ok=True)

summaries = scorecard.summarize_by_label(all_rows)
category_summaries = {
    "baseline": scorecard.summarize_by_category(all_rows, "baseline"),
    "fine_tuned": scorecard.summarize_by_category(all_rows, "fine_tuned"),
}
narrative = scorecard.build_narrative(summaries, bias_summary, n_val=len(val_records))

scorecard.export_markdown(
    drive_run_dir / "scorecard.md", summaries, category_summaries, narrative,
    bias_summary, manifest.__dict__,
)
scorecard.export_csv(drive_run_dir / "metricas_por_registro.csv", all_rows)

(drive_run_dir / "run_manifest.json").write_text(
    _json.dumps(manifest.__dict__, indent=2, ensure_ascii=False), encoding="utf-8"
)
(drive_run_dir / "resultados_baseline.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in baseline_results),
    encoding="utf-8",
)
(drive_run_dir / "resultados_finetuned.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in finetuned_results),
    encoding="utf-8",
)
(drive_run_dir / "position_bias_probe.json").write_text(
    _json.dumps(position_bias_report.__dict__, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Resultados guardados en: {drive_run_dir}")
print((drive_run_dir / "scorecard.md").read_text(encoding="utf-8"))


## (Opcional) Explorar resultados en pandas


In [ ]:
import pandas as pd

df = pd.DataFrame([r.__dict__ for r in all_rows])
df.groupby("label")[["similarity_pct", "judge_composite", "citation_count"]].mean()
